Load the search yaml


In [6]:
from nam.utils.config import load_search_config

config_set, search_space = load_search_config(r"C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas_search.yaml")

print(config_set)
print(search_space)





{'dataset_path': 'C:\\Users\\maart\\OneDrive\\Documenten\\Universiteit\\Scriptie\\python_repo\\thesis-nam\\datasets\\raw\\compas-scores-two-years.csv', 'num_units': 64, 'hidden_sizes': [64, 32], 'decay_rate': 0.995, 'batch_size': 1024, 'num_epochs': 100, 'patience': 50, 'val_check_interval': 10, 'task': 'classification', 'val_frac': 0.15, 'test_frac': 0.15, 'n_trials': 50, 'seed': 42}
{'activation': {'type': 'categorical', 'choices': ['relu']}, 'lr': {'type': 'float_log', 'low': 0.001, 'high': 0.1}, 'output_regularization': {'type': 'float_log', 'low': 0.001, 'high': 0.1}, 'l2_regularization': {'type': 'float_log', 'low': 1e-06, 'high': 0.0001}, 'dropout': {'type': 'categorical', 'choices': [0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]}, 'feature_dropout': {'type': 'categorical', 'choices': [0, 0.05, 0.1, 0.2]}}


Load the data in Optuna

In [7]:
import optuna
from nam.models.nam import NAM
from nam.data.data_utils import load_compas, preprocess, split
from torch.utils.data import DataLoader
from nam.data.dataset import NAMDataset
from nam.training.trainer import Trainer
from pathlib import Path

n_trials = 50 

# --- Data ---
df = load_compas(config_set['dataset_path'])
X, y, feature_names = preprocess(df)
X_train, X_val, X_test, y_train, y_val, y_test = split(
    X, y, config_set["val_frac"], config_set["test_frac"], config_set["seed"]
)

train_loader = DataLoader(NAMDataset(X_train, y_train), batch_size=config_set["batch_size"], shuffle=True)
val_loader   = DataLoader(NAMDataset(X_val,   y_val),   batch_size=config_set["batch_size"], shuffle=False)
test_loader  = DataLoader(NAMDataset(X_test,  y_test),  batch_size=config_set["batch_size"], shuffle=False)

def suggest_hyperparams(trial, search_space: dict) -> dict:
    params = {}
    for name, spec in search_space.items():
        if spec["type"] == "float_log":
            params[name] = trial.suggest_float(name, spec["low"], spec["high"], log=True)
        elif spec["type"] == "float":
            params[name] = trial.suggest_float(name, spec["low"], spec["high"])
        elif spec["type"] == "categorical":
            params[name] = trial.suggest_categorical(name, spec["choices"])
        elif spec["type"] == "int":
            params[name] = trial.suggest_int(name, spec["low"], spec["high"])
    return params


def objective(trial):
    trial_params = suggest_hyperparams(trial, search_space)
    
    model = NAM(
        num_features=X_train.shape[1],
        num_units=config_set["num_units"],
        hidden_sizes=config_set["hidden_sizes"],
        dropout=trial_params["dropout"],
        feature_dropout=trial_params["feature_dropout"],
        activation=trial_params["activation"],
    )

    trainer = Trainer(
        model=model,
        lr=trial_params["lr"],
        decay_rate=config_set["decay_rate"],
        output_regularization=trial_params["output_regularization"],
        l2_regularization=trial_params["l2_regularization"],
        task=config_set["task"],
        num_epochs=config_set["num_epochs"],
        patience=config_set["patience"],
        val_check_interval=config_set["val_check_interval"],
    )

    trainer.train(train_loader, val_loader, trial)
    return trainer.best_val_metric



direction = "maximize" if config_set["task"] == "classification" else "minimize"
dataset_name = Path(config_set["dataset_path"]).stem

storage = f"sqlite:///runs/{dataset_name}_optuna.db"
study_name = f"{dataset_name}_search"

study = optuna.create_study(
    study_name=study_name,
    storage=storage,
    load_if_exists=True,
    direction=direction
)
study.optimize(objective, n_trials=config_set["n_trials"])

print(study.best_trial.value)   # best metric achieved
print(study.best_trial.params)  # dict of best hyperparameters

[I 2026-05-30 01:17:48,443] Using an existing study with name 'compas-scores-two-years_search' instead of creating a new one.


Epoch 1/100| Epoch loss = 0.7186 | best=0.0000


[I 2026-05-30 01:17:52,096] Trial 31 pruned. 


Epoch 1/100| Epoch loss = 0.7245 | best=0.0000


[I 2026-05-30 01:17:55,248] Trial 32 pruned. 


...

Epoch 1/100| Epoch loss = 0.7184 | best=0.0000


[I 2026-05-30 01:23:31,105] Trial 79 pruned. 


Epoch 1/100| Epoch loss = 0.6990 | best=0.0000


[I 2026-05-30 01:23:33,848] Trial 80 pruned. 


0.7383612628043913
{'activation': 'relu', 'lr': 0.007704545276127354, 'output_regularization': 0.023166281131917488, 'l2_regularization': 9.927916301446926e-05, 'dropout': 0, 'feature_dropout': 0.2}


Convert final output back to YAML

In [8]:
import yaml

# Merge fixed params with best trial params
best_params = {**config_set, **study.best_trial.params}

# Remove tuning-only keys not part of NAMConfig
best_params.pop("n_trials", None)

output_path = Path(r"C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs") / f"{dataset_name}_tuned.yaml"

with open(output_path, "w") as f:
    yaml.dump(best_params, f, default_flow_style=False, sort_keys=False)

print(f"Best config saved to {output_path}")

Best config saved to C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas-scores-two-years_tuned.yaml
